In [1]:
import pandas as pd
import pickle
import numpy as np
import category_encoders as ce
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import copy
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import Normalizer
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer

In [2]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
df = pd.read_pickle("Data/df_before_scalling.pickle")

In [4]:
#df_sample = copy.deepcopy(df[:10000])

In [5]:
#let's one-hot-encode for procedure, type contract, central purchasing, eu_fund, framework or dynamic purchasing, 
base_n_encoder_cols = ["cpv_code", "country", "language"]
one_hot_columns = ["type_contract", "procedure", "joint_procurement_involved", "central_purchasing", "eu_fund", "framework or dynamic purchasing", "contracting authority type", "contracting authority activity"]
numerical_columns = ["duration", "nb_tenders_received", "nb_tenders_received_sme", "ac_price_weighting: 0"]
embedding_columns = ["short_descr", "ac criteria", "object_contract/title", "object descr titles", "ac cost criteria"]
categorical_columns = base_n_encoder_cols + one_hot_columns

In [6]:
unprocessed_cols = list(set(df.columns) - set(base_n_encoder_cols) - set(one_hot_columns) - set(numerical_columns) - set(embedding_columns))
unprocessed_cols

['original index', 'award_contract/val_total: 0', 'date_conclusion_contract']

In [7]:
#create a dict where we store the columns that have missing values
has_na_cols = []
for key in pd.isna(df).any().keys():
    if pd.isna(df).any()[key] == True and key != "date_conclusion_contract":
        has_na_cols.append(key)

has_na_cols

['eu_fund',
 'nb_tenders_received',
 'nb_tenders_received_sme',
 'duration',
 'contracting authority type',
 'contracting authority activity',
 'ac_price_weighting: 0']

In [8]:
def train_test_split(df, train_size):
    df = df.sort_values(by = ["date_conclusion_contract"], axis = 0, ascending = True)
    train_set, test_set = np.split(df, [int(train_size * len(df))])
    return train_set, test_set

In [9]:
train_set, test_set = train_test_split(df.drop(columns = ["award_contract/val_total: 0"]), 0.8)
train_test_data_dict = {"train": train_set, "test": test_set}

c:\Users\gbolton\OneDrive\Thesis\.venv2\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [10]:
#for training the randomforest imputer, we need to exclude award_contract/val_total to prevent data leakage. We should also leave out the sentence embeddings and date_conclusion_contract
data_set_impute_split = {"train": {"train_train": [], "train_test": []},
                                   "test": {"test_train": [], "test_test": []}}

drop_columns = embedding_columns + ["date_conclusion_contract"]

for outer_key in data_set_impute_split.keys():
    inner_keys = list(data_set_impute_split[outer_key].keys())
    train_key = inner_keys[0]
    test_key = inner_keys[1]
    #print(inner_keys,"\n", train_key, "\n", test_key)

    data = copy.deepcopy(train_test_data_dict[outer_key])
    train, test = train_test_split(data, 0.8)
    data_set_impute_split[outer_key][train_key] = train.drop(columns = drop_columns)
    data_set_impute_split[outer_key][test_key] = test.drop(columns = drop_columns)

data_set_impute_split_original = copy.deepcopy(data_set_impute_split)

c:\Users\gbolton\OneDrive\Thesis\.venv2\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [22]:
#impute the categorical and numeric columns for all datasets
cat_imputer = SimpleImputer(strategy="most_frequent", missing_values=np.nan)
num_imputer = KNNImputer(n_neighbors=5, missing_values=np.nan)
scaler = StandardScaler()

for outer_key in data_set_impute_split.keys():
    inner_keys = list(data_set_impute_split[outer_key].keys())
    train_key = inner_keys[0]
    test_key = inner_keys[1]

    for col in has_na_cols:
        if col == "ac_price_weighting: 0":
            if col in categorical_columns:
                #reshape the data for each col in the train set
                categorical_data_train = data_set_impute_split[outer_key][train_key][col].values.reshape(-1,1)
                imputed_data_train = cat_imputer.fit_transform(categorical_data_train)
                data_set_impute_split[outer_key][train_key][col] = imputed_data_train.flatten()
                #reshape the data for each col in the test set
                categorical_data_test = data_set_impute_split[outer_key][test_key][col].values.reshape(-1,1)
                imputed_data_test = cat_imputer.fit_transform(categorical_data_test)
                data_set_impute_split[outer_key][test_key][col] = imputed_data_test.flatten()
            elif col in numerical_columns:
                print(col)
                #impute numeric data for each col in the train set
                numeric_data_train = data_set_impute_split[outer_key][train_key][col].values.reshape(-1,1)
                numeric_data_imputed_train = num_imputer.fit_transform(numeric_data_train)
                numeric_data_imputed_scaled_train = scaler.fit_transform(numeric_data_imputed_train)
                data_set_impute_split[outer_key][train_key][col] = numeric_data_imputed_scaled_train
                #impute numeric data for each col in the test set
                numeric_data_test = data_set_impute_split[outer_key][test_key][col].values.reshape(-1,1)
                numeric_data_imputed_test = num_imputer.fit_transform(numeric_data_test)
                numeric_data_imputed_scaled_test = scaler.fit_transform(numeric_data_imputed_test)
                data_set_impute_split[outer_key][test_key][col] = numeric_data_imputed_scaled_test
            else:
                continue

ac_price_weighting: 0
ac_price_weighting: 0


In [24]:
#with open("Data/data_set_impute_split.pickle", "wb") as f:
#    pickle.dump(data_set_impute_split, f)

In [3]:
with open("Data/data_set_impute_split.pickle", "rb") as f:
    data_set_impute_split = pickle.load(f)

In [ ]:
data_set_impute_split

In [12]:
def match_columns(df_train, df_test):
    #handle difference in columns between train and test set
    for col in df_train.columns:
        if col not in df_test.columns:
            df_test[col] = 0

    for col in df_test.columns:
        if col not in df_train.columns:
            df_train[col] = 0
    
    return df_train, df_test

In [14]:
def encode_categorical(target_column: str, df_train, df_test, one_hot_columns = one_hot_columns, base_n_encoder_cols = base_n_encoder_cols):
    #define target column
    target_column = target_column
    
    #initialize encoder
    base_n_encoder_cols = [col for col in base_n_encoder_cols if target_column != col]
    encoder = ce.BaseNEncoder(cols=base_n_encoder_cols, return_df=True, base=2)

    #one-hot-encode categorical columns but remove the target_columns
    one_hot_columns_no_target = [col for col in one_hot_columns if col != target_column]
    df_train = pd.get_dummies(df_train, columns = one_hot_columns_no_target, drop_first=True, dtype = int)
    df_test = pd.get_dummies(df_test, columns = one_hot_columns_no_target, drop_first=True, dtype = int)

    #encode categorical columns with many unique values with the baseNencoder
    df_train = encoder.fit_transform(df_train)
    df_test = encoder.fit_transform(df_test)

    df_train, df_test = match_columns(df_train, df_test)
    
    return df_train, df_test

In [15]:
#create a copy of the dataframe and impute all columns and per column that needs imputation, take the original column as target variable
models_dict = {}
model_results = {}
predict_outputs_dict = {}
predict_outputs_indices_dict = {}
df_final = copy.deepcopy(df)

for outer_key in data_set_impute_split.keys():
    inner_keys = list(data_set_impute_split[outer_key].keys())
    train_key = inner_keys[0]
    test_key = inner_keys[1]
    
    for col in has_na_cols:
        if col not in numerical_columns:
            print(col)
            target_column = col
            #instantie training dataframe with target column
            df_train = copy.deepcopy(data_set_impute_split[outer_key][train_key])
            df_train[target_column] = data_set_impute_split_original[outer_key][train_key][target_column].tolist() #instantiate target column with original data
            #instantie test dataframe with target column
            df_test = copy.deepcopy(data_set_impute_split[outer_key][test_key])
            df_test[target_column] = data_set_impute_split_original[outer_key][test_key][target_column].tolist()
            
            #encode
            df_train, df_test = encode_categorical(target_column=target_column, df_train = df_train, df_test = df_test)
            #initialize the model
            random_forest = RandomForestClassifier(random_state= 1, n_estimators=200)
            #initialize columns used for prediction
            input_columns = [col for col in df_train if col != target_column and col != "original index"] #colums from train and test are equal so pick one
            #initialize the training data
            x_train = df_train.loc[pd.isna(df_train[target_column]) == False][input_columns].values.tolist()
            y_train = df_train.loc[pd.isna(df_train[target_column]) == False][target_column].values.tolist()

            #initialize the testing data
            x_test = df_test.loc[pd.isna(df_test[target_column]) == False][input_columns].values.tolist()
            y_test = df_test.loc[pd.isna(df_test[target_column]) == False][target_column].values.tolist()

            #fit model, add to model dict and predict
            random_forest.fit(x_train, y_train)
            models_dict[str(outer_key + col)] = random_forest
            y_predict = random_forest.predict(x_test)

            #calculate accuracy, and add accuracy to the model
            accuracy = accuracy_score(y_test, y_predict)
            model_results[str(outer_key + ": " + col)] = accuracy

            #initialize dataset by combining test with train
            df_combined = pd.concat([df_train, df_test])
            
            #get indices of those rows for which we will perform prediction
            predict_indices_list = df_combined.loc[pd.isna(df_combined[target_column]) == True].index.tolist()
            #get the input columns for the rows for which we will perform prediciton
            predict_input = df_combined.loc[pd.isna(df_combined[target_column]) == True][input_columns].values.tolist()
            predict_output = random_forest.predict(predict_input)
            
            #join results as a list to the column key
            predict_outputs_dict[target_column] = predict_output
            predict_outputs_indices_dict[target_column] = predict_indices_list

    for column in predict_outputs_indices_dict.keys():
        for index_values, index_df in enumerate(predict_outputs_indices_dict[column]):
            df_final.at[index_df, column] = predict_outputs_dict[column][index_values]

eu_fund


KeyboardInterrupt: 

In [16]:
#split the imputed dataset
df_train, df_test = train_test_split(df_final, 0.8)
#encode the imputed dataset
df_train, df_test = encode_categorical(df_train = df_train, df_test = df_test, target_column="original index")

c:\Users\gbolton\OneDrive\Thesis\.venv2\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
df_train

In [17]:
final_imputer_columns = list(set(df_train.columns) - set(embedding_columns))
#final_imputer_columns.remove("date_conclusion_contract")

#impute and scale the numerical columns with the improved dataset
num_imputer = KNNImputer(n_neighbors=5, missing_values=np.nan)
scaler = StandardScaler()

df_train[final_imputer_columns] = num_imputer.fit_transform(df_train[final_imputer_columns])
df_train[numerical_columns] = scaler.fit_transform(df_train[numerical_columns])

df_test[final_imputer_columns] = num_imputer.fit_transform(df_test[final_imputer_columns])
df_test[numerical_columns] = scaler.fit_transform(df_test[numerical_columns])

ValueError: list.remove(x): x not in list

In [ ]:
print(pd.isna(df_train).any(), pd.isna(df_test).any())

NameError: name 'df_train' is not defined

In [ ]:
#with open("Data/df_preprocessed_train.pickle", "wb") as f:
#    pickle.dump(df_train, f)

#with open("Data/df_preprocessed_test.pickle", "wb") as f:
#    pickle.dump(df_test, f)

---------------------------------------------------------------------------------------------------------------------------------------
PREPROCESSING without encoding
---------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------------------------------------------

In [16]:
print(pd.isna(df_final).any())

original index                     False
language                           False
country                            False
short_descr                        False
ac criteria                        False
object_contract/title              False
object descr titles                False
ac cost criteria                   False
cpv_code                           False
type_contract                      False
procedure                          False
joint_procurement_involved         False
central_purchasing                 False
eu_fund                            False
framework or dynamic purchasing    False
nb_tenders_received                 True
nb_tenders_received_sme             True
award_contract/val_total: 0        False
duration                            True
contracting authority type         False
contracting authority activity     False
date_conclusion_contract            True
dtype: bool


In [19]:
#split the imputed dataset
df_train, df_test = train_test_split(df_final, 0.8)

c:\Users\gbolton\OneDrive\Thesis\.venv2\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [20]:
final_imputer_columns = list(set(df_train.columns) - set(embedding_columns))
final_imputer_columns.remove("date_conclusion_contract")

#impute and scale the numerical columns with the improved dataset
num_imputer = KNNImputer(n_neighbors=5, missing_values=np.nan)
scaler = StandardScaler()

df_train[numerical_columns] = num_imputer.fit_transform(df_train[numerical_columns])
df_train[numerical_columns] = scaler.fit_transform(df_train[numerical_columns])

df_test[numerical_columns] = num_imputer.fit_transform(df_test[numerical_columns])
df_test[numerical_columns] = scaler.fit_transform(df_test[numerical_columns])

In [21]:
with open("Data/df_preprocessed_no_encoding_train.pickle", "wb") as f:
    pickle.dump(df_train, f)

with open("Data/df_preprocessed_no_encoding_test.pickle", "wb") as f:
    pickle.dump(df_test, f)

In [19]:
#THIS PIECE OF CODE WAS USED TO CHEKC IF VALUES WERE THE SAME, AND THEY ARE

#for column in [col for col in has_na_cols if col not in numerical_columns]:
#    if pd.isna(df_original[column]).sum() == len(predict_outputs_dict[column]):
#        print("for column {}, the amount of missing values is equal to the length of predicted values".format(column))
#    else:
#        print(pd.isna(df_original[column]).sum(), len(predict_input), len(predict_outputs_dict[column]))

for column contracting authority type, the amount of missing values is equal to the length of predicted values
for column contracting authority activity, the amount of missing values is equal to the length of predicted values


In [ ]:
#1. encode the train/test without target variable.
#2. get all rows of the encoded train/test where value for target is nan
#3. predict for these rows and save in dicts of dicts where outer key = column and inner key=index
#4. when done with all predictions, merge predicted values with original dataset

In [17]:
#use the model to predict the values 
#retrieve rows for which no value exists

{'train: eu_fund': 0.9070094998351763,
 'train: contracting authority type': 0.1686178808165419,
 'train: contracting authority activity': 0.4151851974009697,
 'test: eu_fund': 0.9147788565264293,
 'test: contracting authority type': 0.54656832104013,
 'test: contracting authority activity': 0.5073768442110528}

In [18]:
results

{'train: eu_fund': 0.9475,
 'train: contracting authority type': 0.18422841604350781,
 'train: contracting authority activity': 0.2794017675050986,
 'test: eu_fund': 0.925,
 'test: contracting authority type': 0.20855614973262032,
 'test: contracting authority activity': 0.5213903743315508}

In [19]:
with open("Data/imputing_model_results.pickle", "wb") as f:
    pickle.dump(results, f)

In [20]:
with open("Data/imputing_models.pickle", "wb") as f:
    pickle.dump(models_dict, f)

KeyboardInterrupt: 